# Model Exploration

This notebook documents the exploratory process that led to our final model architecture in `src/train.py`. We evaluate LightGBM as our framework, prototype hyperparameter tuning with Optuna, and validate with time-series cross-validation.

**Objectives:**
1. Explore price trends and return distributions across tickers
2. Analyze feature-target correlations from the engineered feature set
3. Train and validate a dual-objective LightGBM framework (classifier + regressor)
4. Demonstrate the signal generation pipeline

In [ ]:
import polars as pl
import numpy as np
import plotly.express as px
import plotly.io as pio
import lightgbm as lgb
import optuna
import joblib
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, root_mean_squared_error
from sklearn.model_selection import TimeSeriesSplit

pio.templates.default = "plotly_white"
optuna.logging.set_verbosity(optuna.logging.WARNING)

## 1. EDA: Price History and Return Distribution

In [ ]:
# Load raw prices for multi-ticker EDA
df_raw = pl.read_parquet("../data/raw/prices.parquet")
print(f"Raw prices: {df_raw.shape}")

fig_price = px.line(df_raw.to_pandas(), x="Date", y="Close", color="Ticker",
                    title="Historical Close Prices",
                    labels={"Close": "Price ($)", "Date": ""})
fig_price.show()

In [ ]:
# Daily return distribution: heavy tails indicate high-risk periods
df_returns = df_raw.with_columns(
    (pl.col("Close") / pl.col("Close").shift(1).over("Ticker") - 1).alias("Return")
).drop_nulls()

fig_dist = px.histogram(df_returns.to_pandas(), x="Return", color="Ticker", marginal="box",
                        title="Distribution of Daily Returns", barmode="overlay",
                        labels={"Return": "Daily Percentage Change"})
fig_dist.show()

## 2. Feature-Target Correlation

Load the processed feature set (output of `src/etl.py`) and examine how engineered features correlate with our targets.

In [ ]:
# Load engineered features from ETL output
df = pl.read_parquet("../data/processed/features.parquet").sort("Date")
print(f"Feature set: {df.shape}")

# Correlation heatmap with production feature names
corr_cols = ["Dist_SMA_20", "Dist_SMA_50", "Dist_SMA_200", "RSI_Proxy",
             "Vol_20", "ATR_14", "ROA", "Net_Margin", "Target_Log_Return"]
corr_matrix = df.select(corr_cols).to_pandas().corr()

fig_corr = px.imshow(corr_matrix, text_auto=".2f", aspect="auto",
                     title="Feature Correlation Matrix",
                     color_continuous_scale="RdBu_r")
fig_corr.show()

## 3. Train/Test Preparation

Extract features and targets from the processed dataset. The `get_feature_list()` helper automatically selects all numeric columns except metadata and targets, ensuring the model sees only valid predictors.

In [ ]:
# Replicate get_feature_list() logic from src/train.py
exclude = ["Ticker", "Date", "SimFinId", "Target_Is_Up", "Target_Log_Return"]
numeric_cols = [c for c, t in df.schema.items() if t in [pl.Int64, pl.Float64, pl.Int32, pl.Float32]]
features = [c for c in numeric_cols if c not in exclude]

X = df.select(features).to_pandas().values
y_class = df["Target_Is_Up"].to_pandas().values
y_reg = df["Target_Log_Return"].to_pandas().values

print(f"Features: {len(features)}")
print(f"Sample names: {features[:5]}")
print(f"X shape: {X.shape}, y_class shape: {y_class.shape}, y_reg shape: {y_reg.shape}")

## 4. Baseline: LightGBM with Default Parameters

Before tuning, we establish a baseline using default LightGBM settings with `TimeSeriesSplit` (5 folds) to respect temporal ordering. The classifier is evaluated on **precision**, since we prefer fewer, higher-confidence buy signals over high recall.

In [ ]:
# Baseline classifier with TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=5)
baseline_precisions = []
baseline_rmses = []

for train_idx, val_idx in tscv.split(X):
    X_train, X_val = X[train_idx], X[val_idx]

    # Classification baseline
    clf = lgb.LGBMClassifier(verbosity=-1)
    clf.fit(X_train, y_class[train_idx])
    preds_cls = clf.predict(X_val)
    baseline_precisions.append(precision_score(y_class[val_idx], preds_cls, zero_division=0))

    # Regression baseline
    reg = lgb.LGBMRegressor(verbosity=-1)
    reg.fit(X_train, y_reg[train_idx])
    preds_reg = reg.predict(X_val)
    baseline_rmses.append(root_mean_squared_error(y_reg[val_idx], preds_reg))

print(f"Baseline Classifier Precision (5-fold CV): {np.mean(baseline_precisions):.4f}")
print(f"Baseline Regressor RMSE (5-fold CV): {np.mean(baseline_rmses):.6f}")

## 5. Hyperparameter Tuning with Optuna

We use Optuna's Bayesian search to find optimal hyperparameters. Here we run a reduced study (10 trials) to demonstrate the workflow; the production script (`src/train.py`) runs 30 trials per model.

In [ ]:
# Optuna objective for classification (maximize precision)
def objective_classification(trial):
    param = {
        "objective": "binary", "metric": "binary_logloss", "verbosity": -1,
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 50, 500),
        "num_leaves": trial.suggest_int("num_leaves", 20, 100),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 5, 100),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.4, 1.0),
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
    }
    tscv = TimeSeriesSplit(n_splits=5)
    precisions = []
    for train_idx, val_idx in tscv.split(X):
        clf = lgb.LGBMClassifier(**param)
        clf.fit(X[train_idx], y_class[train_idx])
        preds = clf.predict(X[val_idx])
        precisions.append(precision_score(y_class[val_idx], preds, zero_division=0))
    return np.mean(precisions)

# Optuna objective for regression (minimize RMSE)
def objective_regression(trial):
    param = {
        "objective": "regression", "metric": "rmse", "verbosity": -1,
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 50, 500),
        "num_leaves": trial.suggest_int("num_leaves", 20, 100),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 5, 100),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.4, 1.0),
    }
    tscv = TimeSeriesSplit(n_splits=5)
    rmses = []
    for train_idx, val_idx in tscv.split(X):
        reg = lgb.LGBMRegressor(**param)
        reg.fit(X[train_idx], y_reg[train_idx])
        preds = reg.predict(X[val_idx])
        rmses.append(root_mean_squared_error(y_reg[val_idx], preds))
    return np.mean(rmses)

# Run reduced Optuna studies (10 trials each, production uses 30)
study_clf = optuna.create_study(direction="maximize")
study_clf.optimize(objective_classification, n_trials=10)

study_reg = optuna.create_study(direction="minimize")
study_reg.optimize(objective_regression, n_trials=10)

print(f"Best Classifier Precision: {study_clf.best_value:.4f}")
print(f"Best Classifier Params: {study_clf.best_params}\n")
print(f"Best Regressor RMSE: {study_reg.best_value:.6f}")
print(f"Best Regressor Params: {study_reg.best_params}")

## 6. Final Models and Feature Importance

Train the final classifier and regressor using the best parameters found by Optuna, then inspect which features the classifier relies on most.

In [ ]:
# Train final models on all data with best params
best_clf = lgb.LGBMClassifier(**study_clf.best_params, verbosity=-1)
best_clf.fit(X, y_class)

best_reg = lgb.LGBMRegressor(**study_reg.best_params, verbosity=-1)
best_reg.fit(X, y_reg)

# Feature importance (top 10 for classifier)
lgb.plot_importance(best_clf, max_num_features=10, importance_type="gain",
                    figsize=(8, 5), title="Top 10 Features (Classifier — Gain)")
plt.tight_layout()
plt.show()

## 7. Signal Generation Demo

Demonstrate the prediction pipeline on the latest available data point. The classifier outputs `prob_up`, the regressor outputs `pred_log_return`, and the signal logic maps these to BUY / HOLD / SELL with a target price.

In [ ]:
# Grab the latest row from the feature set as a live inference example
X_latest = df.select(features).tail(1).to_pandas().values
latest_row = df.tail(1)

prob_up = float(best_clf.predict_proba(X_latest)[0][1])
pred_log_return = float(best_reg.predict(X_latest)[0])

current_price = latest_row["Close"].item()
target_price = current_price * np.exp(pred_log_return)

if prob_up > 0.55:
    signal = "BUY"
elif prob_up < 0.45:
    signal = "SELL"
else:
    signal = "HOLD"

ticker = latest_row["Ticker"].item()
date = latest_row["Date"].item()

print(f"Ticker: {ticker} | Date: {date}")
print(f"Current Price: ${current_price:.2f}")
print(f"Prob Up: {prob_up:.4f}")
print(f"Predicted Log Return: {pred_log_return:.6f}")
print(f"Target Price: ${target_price:.2f}")
print(f"Signal: {signal}")

## Next Steps

This exploration was formalized into two production modules:

- **`src/train.py`**: Runs 30 Optuna trials per model (vs. 10 here), saves trained models and feature lists to `models/`
- **`src/predict.py`**: Fetches live data via the SimFin API, replicates the ETL feature engineering for a single ticker, and generates real-time trading signals